<a href="https://colab.research.google.com/github/rfroiman/DI_Bootcamp/blob/main/Week8/Day4/Pinecone_Serverless_Reranking_Daily_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q "pinecone==6.0.1" pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 2.6 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
from pinecone import Pinecone

# Load API key securely from Colab Secrets
api_key = userdata.get("PINECONE_API_KEY")

# Initialize Pinecone client
pc = Pinecone(api_key=api_key)

print("Pinecone authentication successful!")

Pinecone authentication successful!


In [3]:
# Define the search query
query = "Tell me about Apple's products"

# Create documents with different meanings of Apple
documents = [
    "Apples are nutritious fruits rich in fiber and vitamins.",
    "Apple develops the iPhone, iPad, MacBook, and Apple Watch.",
    "Green and red apples are commonly used in pies and juices.",
    "Apple offers products such as AirPods, iMac, and Mac mini.",
    "Apple provides software services including iCloud and Apple Music."
]

print("Query:", query)
print("Total documents:", len(documents))

Query: Tell me about Apple's products
Total documents: 5


In [4]:
# Rerank documents based on semantic relevance
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[
        {"id": str(i), "text": doc}
        for i, doc in enumerate(documents)
    ],
    top_n=3,
    return_documents=True
)

print("Reranking completed successfully!")

Reranking completed successfully!


In [5]:
# Display reranked results
def show_reranked_results(query, matches):
    print(f"Query: {query}\n")

    for i, match in enumerate(matches):
        print(f"Rank: {i + 1}")
        print(f"Score: {match.score:.4f}")
        print(f"Document: {match.document.text}")
        print("-" * 50)

show_reranked_results(query, reranked.data)

Query: Tell me about Apple's products

Rank: 1
Score: 0.9216
Document: Apple offers products such as AirPods, iMac, and Mac mini.
--------------------------------------------------
Rank: 2
Score: 0.7326
Document: Apple develops the iPhone, iPad, MacBook, and Apple Watch.
--------------------------------------------------
Rank: 3
Score: 0.2892
Document: Apple provides software services including iCloud and Apple Music.
--------------------------------------------------


In [6]:
from pinecone import ServerlessSpec

# Configure cloud provider and region
cloud = "aws"
region = "us-east-1"

# Define serverless specifications
spec = ServerlessSpec(
    cloud=cloud,
    region=region
)

# Define index name
index_name = "medical-notes-index"

print("Cloud:", cloud)
print("Region:", region)
print("Index:", index_name)

Cloud: aws
Region: us-east-1
Index: medical-notes-index


In [7]:
# Create the index only if it does not exist
if not pc.has_index(name=index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=spec
    )
    print("Index created successfully!")
else:
    print("Index already exists!")

print("Index name:", index_name)

Index created successfully!
Index name: medical-notes-index


In [8]:
import pandas as pd
import requests
import io

# Download sample medical notes
dataset_url = (
    "https://raw.githubusercontent.com/devtlv/"
    "Datasets-GEN-AI-Bootcamp/refs/heads/main/"
    "Week%208/W8D2/data.json"
)

response = requests.get(dataset_url, timeout=60)
response.raise_for_status()

# Load JSON Lines into a DataFrame
df = pd.read_json(
    io.BytesIO(response.content),
    orient="records",
    lines=True
)

print("Data shape:", df.shape)
display(df.head())

Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


In [9]:
# Connect to the Pinecone index
index = pc.Index(name=index_name)

# Upload medical notes and their embeddings
index.upsert_from_dataframe(df)

print("Medical notes uploaded successfully!")

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

Medical notes uploaded successfully!


In [10]:
# Check index statistics
stats = index.describe_index_stats()

print("Total vectors:", stats.total_vector_count)
print("Index statistics:", stats)

Total vectors: 100
Index statistics: {'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}


In [12]:
%pip install -q transformers torch

In [13]:
import torch
from transformers import AutoTokenizer, AutoModel

# Load the embedding model
model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

print("Embedding model loaded successfully!")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [14]:
# Convert text into a 384-dimensional embedding
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # Mean pooling with attention mask
    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    embedding = (
        (token_embeddings * attention_mask).sum(dim=1)
        / attention_mask.sum(dim=1)
    )

    # Normalize the embedding
    embedding = torch.nn.functional.normalize(embedding, p=2, dim=1)

    return embedding[0].tolist()


# Test the embedding function
test_embedding = get_embedding("Patient experiencing chest pain")

print("Embedding dimensions:", len(test_embedding))

Embedding dimensions: 384


In [15]:
# Define a medical search query
medical_query = "Patient experiencing chest pain and heart problems"

# Generate query embedding
query_embedding = get_embedding(medical_query)

# Search the Pinecone index
search_results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

print("Query:", medical_query)
print("Matches found:", len(search_results.matches))

Query: Patient experiencing chest pain and heart problems
Matches found: 5


In [16]:
# Display retrieved medical records
for i, match in enumerate(search_results.matches, start=1):
    print(f"Rank: {i}")
    print(f"Patient ID: {match.id}")
    print(f"Similarity Score: {match.score:.4f}")
    print(f"Medical Notes: {match.metadata}")
    print("-" * 60)

Rank: 1
Patient ID: P001
Similarity Score: 0.7213
Medical Notes: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}
------------------------------------------------------------
Rank: 2
Patient ID: P016
Similarity Score: 0.4994
Medical Notes: {'condition': 'heart murmur', 'referral': 'cardiology'}
------------------------------------------------------------
Rank: 3
Patient ID: P0100
Similarity Score: 0.4184
Medical Notes: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}
------------------------------------------------------------
Rank: 4
Patient ID: P090
Similarity Score: 0.4011
Medical Notes: {'advice': 'stress management', 'symptoms': 'stress, burnout'}
------------------------------------------------------------
Rank: 5
Patient ID: P042
Similarity Score: 0.4011
Medical Notes: {'advice': 'stress management', 'symptoms': 'stress, burnout'}
------------------------------------------------------------


In [17]:
# Prepare retrieved medical records for reranking
transformed_documents = []

for match in search_results.matches:
    medical_text = ", ".join(
        f"{key}: {value}"
        for key, value in match.metadata.items()
    )

    transformed_documents.append({
        "id": match.id,
        "reranking_field": medical_text
    })

# Display prepared documents
for document in transformed_documents:
    print(document)

{'id': 'P001', 'reranking_field': 'symptoms: chest pain, tests: EKG, stress test'}
{'id': 'P016', 'reranking_field': 'condition: heart murmur, referral: cardiology'}
{'id': 'P0100', 'reranking_field': 'advice: over-the-counter pain relief, stretching, symptoms: muscle pain'}
{'id': 'P090', 'reranking_field': 'advice: stress management, symptoms: stress, burnout'}
{'id': 'P042', 'reranking_field': 'advice: stress management, symptoms: stress, burnout'}


In [18]:
# Rerank medical records by relevance
medical_reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=medical_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True
)

print("Medical reranking completed successfully!")

Medical reranking completed successfully!


In [19]:
# Display final reranked medical records
print("Query:", medical_query)
print("=" * 60)

for i, result in enumerate(medical_reranked.data, start=1):
    print(f"Rank: {i}")
    print(f"Relevance Score: {result.score:.4f}")
    print(f"Medical Record: {result.document.reranking_field}")
    print("-" * 60)

Query: Patient experiencing chest pain and heart problems
Rank: 1
Relevance Score: 0.5398
Medical Record: symptoms: chest pain, tests: EKG, stress test
------------------------------------------------------------
Rank: 2
Relevance Score: 0.0489
Medical Record: condition: heart murmur, referral: cardiology
------------------------------------------------------------
Rank: 3
Relevance Score: 0.0047
Medical Record: advice: over-the-counter pain relief, stretching, symptoms: muscle pain
------------------------------------------------------------


Conclusion — Pinecone Serverless Reranking

This project implemented a semantic search and reranking pipeline using Pinecone Serverless.

A serverless index was created with 384-dimensional vectors and cosine similarity. A dataset containing 100 medical records was uploaded and successfully indexed.

The initial semantic search retrieved five medical records. The bge-reranker-v2-m3 model then reranked these results and selected the three most relevant documents.

The final ranking prioritized records related to chest pain and cardiac conditions, demonstrating how reranking can improve the relevance ordering of retrieved documents.